In [47]:
# Cell 1 — Imports
import warnings
import os
import numpy as np
import pandas as pd
from datetime import datetime

warnings.filterwarnings("ignore")

In [48]:
# Cell 2 — Config (change path here)
FILE_PATH  = "/Users/huntstar/Projects/Zomato_project/Data/zomato.csv"   # ← your real file path
OUTPUT_PATH = "profiling_report_ipy.txt"
TOP_N_FREQ  = 10                # top N frequency values per column

# ── FIX 1: Explicit numeric whitelist (no auto-detection) ──
NUMERIC_COLUMNS = [
    "votes",
    "rate",
    "approx_cost(for two people)"
]

# ── FIX 4: Feature category map ──
FEATURE_CATEGORIES = {
    "url"                        : "Drop",
    "address"                    : "Drop",
    "name"                       : "Identifier",
    "online_order"               : "Binary",
    "book_table"                 : "Binary",
    "rate"                       : "Target Variable",
    "votes"                      : "Numerical",
    "phone"                      : "Drop",
    "location"                   : "Categorical",
    "rest_type"                  : "Categorical",
    "dish_liked"                 : "NLP",
    "cuisines"                   : "Recommendation",
    "approx_cost(for two people)": "Numerical",
    "reviews_list"               : "NLP / Sentiment",
    "menu_item"                  : "RAG",
    "listed_in(type)"            : "Categorical",
    "listed_in(city)"            : "Categorical",
}

# ── FIX 3: Missing value recommendations ──
MISSING_RECOMMENDATIONS = {
    "url"                        : "Drop column",
    "address"                    : "Drop column",
    "name"                       : "Flag if null — critical",
    "online_order"               : "Flag if null — critical",
    "book_table"                 : "Flag if null — critical",
    "rate"                       : "Clean (remove /5, cast to float)",
    "votes"                      : "Fill 0 if null",
    "phone"                      : "Drop column",
    "location"                   : "Impute with mode or flag",
    "rest_type"                  : "Impute with mode or flag",
    "dish_liked"                 : "Keep — high null expected",
    "cuisines"                   : "Impute with mode",
    "approx_cost(for two people)": "Fill median if null",
    "reviews_list"               : "NLP pipeline",
    "menu_item"                  : "RAG pipeline",
    "listed_in(type)"            : "Impute with mode",
    "listed_in(city)"            : "Impute with mode",
}

# ── Invalid sentinels ──
SENTINELS = {
    "nan", "none", "null", "na", "n/a", "n.a", "n.a.",
    "-", "--", "---", "new", "unknown", "undefined",
    "not available", "not applicable", ""
}

In [49]:
# Cell 3 — Load Data
def load_data(path):
    csv_size_mb = os.path.getsize(path) / (1024 ** 2)
    print(f"CSV file size : {csv_size_mb:.2f} MB")

    kwargs = dict(engine="python", on_bad_lines="skip")
    try:
        df = pd.read_csv(path, encoding="utf-8", **kwargs)
    except UnicodeDecodeError:
        print("UTF-8 failed — retrying with latin-1")
        df = pd.read_csv(path, encoding="latin-1", **kwargs)

    # FIX 2: report both csv size and in-memory size
    mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"In-memory size: {mem_mb:.2f} MB  (strings expand because each Python str object has overhead)")
    print(f"Loaded        : {df.shape[0]:,} rows × {df.shape[1]} columns")
    return df, mem_mb

df, MEM_MB = load_data(FILE_PATH)
CSV_MB = os.path.getsize(FILE_PATH) / (1024 ** 2)
df.head(3)

CSV file size : 547.48 MB
In-memory size: 466.79 MB  (strings expand because each Python str object has overhead)
Loaded        : 51,155 rows × 17 columns


,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari


In [50]:
# Cell 4 — Dataset Overview
dup_rows        = int(df.duplicated().sum())
fully_null_rows = int(df.isnull().all(axis=1).sum())
fully_null_cols = [c for c in df.columns if df[c].isnull().all()]

print("=" * 55)
print("  DATASET OVERVIEW")
print("=" * 55)
print(f"  Total rows           : {len(df):,}")
print(f"  Total columns        : {len(df.columns)}")
print(f"  CSV file size        : {CSV_MB:.2f} MB")
print(f"  In-memory size       : {MEM_MB:.2f} MB")
print(f"  Exact duplicate rows : {dup_rows:,}  ({round(dup_rows/len(df)*100,2)}%)")
print(f"  Fully null rows      : {fully_null_rows:,}")
print(f"  Fully null columns   : {fully_null_cols or 'None'}")
print(f"  dtypes               : {df.dtypes.astype(str).value_counts().to_dict()}")

  DATASET OVERVIEW
  Total rows           : 51,155
  Total columns        : 17
  CSV file size        : 547.48 MB
  In-memory size       : 466.79 MB
  Exact duplicate rows : 0  (0.0%)
  Fully null rows      : 0
  Fully null columns   : None
  dtypes               : {'str': 16, 'int64': 1}


In [51]:
# Cell 5 — FIX 3: Missing Value Classification Table
def missing_classification(df):
    rows = []
    for col in df.columns:
        null_c   = int(df[col].isna().sum())
        blank_c  = int(df[col].dropna().astype(str).str.strip().eq("").sum())
        inv_c    = int(df[col].dropna().astype(str).str.strip().str.lower().isin(SENTINELS).sum())
        total_missing = null_c + blank_c + inv_c
        usable   = len(df) - total_missing
        miss_pct = round(total_missing / len(df) * 100, 2)
        rows.append({
            "column"          : col,
            "null"            : null_c,
            "blank"           : blank_c,
            "invalid_sentinel": inv_c,
            "total_missing"   : total_missing,
            "usable_values"   : usable,
            "missing_pct"     : miss_pct,
            "recommendation"  : MISSING_RECOMMENDATIONS.get(col, "Review manually"),
        })
    return pd.DataFrame(rows)

missing_df = missing_classification(df)
missing_df.sort_values("missing_pct", ascending=False)

,column,null,blank,invalid_sentinel,total_missing,usable_values,missing_pct,recommendation
10,dish_liked,28074,0,0,28074,23081,54.88,Keep — high null expected
5,rate,7775,0,2275,10050,41105,19.65,"Clean (remove /5, cast to float)"
7,phone,1203,0,0,1203,49952,2.35,Drop column
12,approx_cost(for two people),344,0,0,344,50811,0.67,Fill median if null
9,rest_type,224,0,0,224,50931,0.44,Impute with mode or flag
11,cuisines,45,0,0,45,51110,0.09,Impute with mode
8,location,21,0,0,21,51134,0.04,Impute with mode or flag
15,listed_in(type),0,0,0,0,51155,0.00,Impute with mode
14,menu_item,0,0,0,0,51155,0.00,RAG pipeline
13,reviews_list,0,0,0,0,51155,0.00,NLP pipeline


In [52]:
# Cell 6 — FIX 4 + FIX 5: Feature Categories + High Cardinality Flags
def feature_category_table(df):
    rows = []
    for col in df.columns:
        unique_c   = int(df[col].nunique())
        card_pct   = round(unique_c / len(df) * 100, 2)
        high_card  = "⚠️ HIGH" if card_pct > 50 else "OK"
        category   = FEATURE_CATEGORIES.get(col, "Unknown")
        action     = "Drop" if category == "Drop" else (
                     "Flag for removal" if high_card == "⚠️ HIGH" and category not in
                     ["NLP / Sentiment", "NLP", "RAG", "Identifier"] else "Keep"
        )
        rows.append({
            "column"        : col,
            "category"      : category,
            "unique_count"  : unique_c,
            "cardinality_pct": card_pct,
            "high_cardinality": high_card,
            "suggested_action": action,
        })
    return pd.DataFrame(rows)

feature_df = feature_category_table(df)
feature_df

,column,category,unique_count,cardinality_pct,high_cardinality,suggested_action
0,url,Drop,51155,100.00,⚠️ HIGH,Drop
1,address,Drop,11485,22.45,OK,Drop
2,name,Identifier,8783,17.17,OK,Keep
3,online_order,Binary,2,0.00,OK,Keep
4,book_table,Binary,2,0.00,OK,Keep
5,rate,Target Variable,64,0.13,OK,Keep
6,votes,Numerical,2235,4.37,OK,Keep
7,phone,Drop,14888,29.10,OK,Drop
8,location,Categorical,93,0.18,OK,Keep
9,rest_type,Categorical,93,0.18,OK,Keep


In [53]:
# Cell 7 — Unique Values & Cardinality (full table)
unique_df = pd.DataFrame({
    "unique_count"   : df.nunique(),
    "cardinality_pct": (df.nunique() / len(df) * 100).round(2)
}).sort_values("cardinality_pct", ascending=False)
unique_df

,unique_count,cardinality_pct
url,51155,100.00
reviews_list,21951,42.91
phone,14888,29.10
address,11485,22.45
menu_item,9030,17.65
name,8783,17.17
dish_liked,5232,10.23
cuisines,2715,5.31
votes,2235,4.37
location,93,0.18


In [54]:
# Cell 8 — FIX 1: Numeric Statistics (whitelist only, no false detection)
def numeric_stats(df, numeric_cols):
    results = {}
    for col in numeric_cols:
        if col not in df.columns:
            continue
        s = df[col].copy()

        # Clean 'rate' column: "4.1/5" → 4.1, "NEW" → NaN
        if s.dtype == object:
            s = s.astype(str).str.extract(r"([\d.]+)", expand=False)
            s = pd.to_numeric(s, errors="coerce")
        else:
            s = pd.to_numeric(s, errors="coerce")

        if s.notna().sum() == 0:
            continue

        desc = s.describe(percentiles=[.05, .25, .5, .75, .95])
        results[col] = {
            "count"    : int(s.notna().sum()),
            "mean"     : round(float(desc["mean"]),  4),
            "std"      : round(float(desc["std"]),   4),
            "min"      : round(float(desc["min"]),   4),
            "p5"       : round(float(desc["5%"]),    4),
            "p25"      : round(float(desc["25%"]),   4),
            "median"   : round(float(desc["50%"]),   4),
            "p75"      : round(float(desc["75%"]),   4),
            "p95"      : round(float(desc["95%"]),   4),
            "max"      : round(float(desc["max"]),   4),
            "zeros"    : int((s == 0).sum()),
            "negatives": int((s < 0).sum()),
        }
    return pd.DataFrame(results).T

numeric_stats_df = numeric_stats(df, NUMERIC_COLUMNS)
numeric_stats_df

,count,mean,std,min,p5,p25,median,p75,p95,max,zeros,negatives
votes,51155.0,260.2334,709.2527,0.0,0.0,6.0,39.0,189.0,1237.0,16832.0,10025.0,0.0
approx_cost(for two people),44236.0,415.7637,194.2514,40.0,150.0,300.0,400.0,500.0,800.0,950.0,0.0,0.0


In [55]:
# Cell 9 — FIX 6: Outlier Detection (IQR method, whitelist columns only)
def outlier_detection(df, numeric_cols):
    rows = []
    for col in numeric_cols:
        if col not in df.columns:
            continue
        s = df[col].copy()

        # Always force to numeric — extract digits first if object dtype
        if s.dtype == object:
            s = pd.to_numeric(
                s.astype(str).str.extract(r"([\d.]+)", expand=False),
                errors="coerce"
            )
        else:
            s = pd.to_numeric(s, errors="coerce")  # catches any strays

        # Drop NaN before IQR — quantile fails on all-NaN or mixed types
        s = s.dropna().astype(float)

        if len(s) == 0:
            continue

        Q1  = s.quantile(0.25)
        Q3  = s.quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers_low  = int((s < lower).sum())
        outliers_high = int((s > upper).sum())
        total_out     = outliers_low + outliers_high
        out_pct       = round(total_out / len(s) * 100, 2)

        rows.append({
            "column"        : col,
            "Q1"            : round(Q1, 4),
            "Q3"            : round(Q3, 4),
            "IQR"           : round(IQR, 4),
            "lower_fence"   : round(lower, 4),
            "upper_fence"   : round(upper, 4),
            "outliers_low"  : outliers_low,
            "outliers_high" : outliers_high,
            "total_outliers": total_out,
            "outlier_pct"   : out_pct,
        })
    return pd.DataFrame(rows)

outlier_df = outlier_detection(df, NUMERIC_COLUMNS)
outlier_df

,column,Q1,Q3,IQR,lower_fence,upper_fence,outliers_low,outliers_high,total_outliers,outlier_pct
0,votes,6.0,189.0,183.0,-268.5,463.5,0,6880,6880,13.45
1,approx_cost(for two people),300.0,500.0,200.0,0.0,800.0,0,900,900,2.03


In [56]:
# Cell 10 — Frequency (top N per column)
for col in df.columns:
    print(f"\n{'='*55}")
    print(f"  COLUMN: {col}  [{FEATURE_CATEGORIES.get(col, 'Unknown')}]")
    print(f"{'='*55}")
    vc = df[col].value_counts(dropna=False).head(TOP_N_FREQ).reset_index()
    vc.columns = ["value", "count"]
    vc["pct"] = (vc["count"] / len(df) * 100).round(2)
    display(vc)


  COLUMN: url  [Drop]


,value,count,pct
0,https://www.zomato.com/bangalore/jalsa-banasha...,1,0.0
1,https://www.zomato.com/bangalore/spice-elephan...,1,0.0
2,https://www.zomato.com/SanchurroBangalore?cont...,1,0.0
3,https://www.zomato.com/bangalore/addhuri-udupi...,1,0.0
4,https://www.zomato.com/bangalore/grand-village...,1,0.0
5,https://www.zomato.com/bangalore/timepass-dinn...,1,0.0
6,https://www.zomato.com/bangalore/rosewood-inte...,1,0.0
7,https://www.zomato.com/bangalore/onesta-banash...,1,0.0
8,https://www.zomato.com/bangalore/penthouse-caf...,1,0.0
9,https://www.zomato.com/bangalore/smacznego-ban...,1,0.0



  COLUMN: address  [Drop]


,value,count,pct
0,Delivery Only,127,0.25
1,"14th Main, 4th Sector, HSR, Bangalore",71,0.14
2,"The Ritz-Carlton, 99, Residency Road, Bangalore",61,0.12
3,"Citrus Hotels, 34, Cunningham Road, Bangalore",53,0.10
4,"Conrad Bengaluru, Kensington Road, Ulsoor, Ban...",49,0.10
5,"710, Thubarahalli, Varthur Main Road, Whitefie...",46,0.09
6,"1, 100 Feet Ring Road, 1st Phase, 2nd Stage, B...",43,0.08
7,"40/2, Lavelle Road, Bangalore",41,0.08
8,"The Park Bangalore, 14/7, MG Road, Bangalore",38,0.07
9,"Radisson Blu, 1, Palace Road, Race Course Road...",37,0.07



  COLUMN: name  [Identifier]


...

,value,count,pct
0,[],39177,76.58
1,"['Avil Milk', 'Oreo Shake', 'Chocolate Shake',...",10,0.02
2,"['Butter Chicken Pizza', 'Bombay Veggie Burger...",10,0.02
3,"['Chicken Cheese Burger', 'Chicken Billys BIg ...",9,0.02
4,"['Students Veg Combo', 'Office Veg Combo', 'St...",9,0.02
5,"['The O.G Waffle', ""S'mores"", 'The Dirty Duche...",8,0.02
6,"['Alfredo Pasta', 'Pesto Pasta', 'Red Sauce Pa...",8,0.02
7,"['Violet Macaron', 'Chocolate Hazelnut Mousse'...",8,0.02
8,"['Chicken Spring Rolls', 'Chicken Picante Pizz...",8,0.02
9,"['Chicken 65', 'Chicken Chilly', 'Chicken Lemo...",8,0.02



  COLUMN: listed_in(type)  [Categorical]


,value,count,pct
0,Delivery,25771,50.38
1,Dine-out,17565,34.34
2,Desserts,3563,6.97
3,Cafes,1690,3.30
4,Drinks & nightlife,1030,2.01
5,Buffet,857,1.68
6,Pubs and bars,679,1.33



  COLUMN: listed_in(city)  [Categorical]


,value,count,pct
0,BTM,3259,6.37
1,Koramangala 7th Block,2901,5.67
2,Koramangala 5th Block,2788,5.45
3,Koramangala 4th Block,2755,5.39
4,Koramangala 6th Block,2577,5.04
5,Jayanagar,2361,4.62
6,JP Nagar,2080,4.07
7,Indiranagar,1849,3.61
8,Church Street,1815,3.55
9,Brigade Road,1754,3.43


In [57]:
# Cell 11 — FIX 7: ML Readiness Report
def ml_readiness(df, missing_df, feature_df, outlier_df):
    target_col    = "rate"
    needs_missing = missing_df[missing_df["missing_pct"] > 0]["column"].tolist()
    needs_encode  = feature_df[feature_df["category"].isin(["Categorical", "Binary"])]["column"].tolist()
    needs_scale   = NUMERIC_COLUMNS
    needs_nlp     = feature_df[feature_df["category"].str.contains("NLP")]["column"].tolist()
    needs_rag     = feature_df[feature_df["category"] == "RAG"]["column"].tolist()
    needs_rec     = feature_df[feature_df["category"] == "Recommendation"]["column"].tolist()
    drop_cols     = feature_df[feature_df["category"] == "Drop"]["column"].tolist()
    high_card     = feature_df[feature_df["high_cardinality"] == "⚠️ HIGH"]["column"].tolist()
    has_outliers  = outlier_df[outlier_df["total_outliers"] > 0]["column"].tolist()

    print("=" * 55)
    print("  ML READINESS REPORT")
    print("=" * 55)
    print(f"\n  TARGET VARIABLE")
    print(f"  ─────────────────────────────────────")
    print(f"  Column                : {target_col}")
    print(f"  Type                  : Continuous (regression) / Binned (classification)")
    print(f"  Ready for modelling   : ❌  Needs cleaning first (rate stored as '4.1/5')")

    print(f"\n  DATA QUALITY")
    print(f"  ─────────────────────────────────────")
    print(f"  Missing value treatment  : {'✅ Required' if needs_missing else '✅ Clean'}")
    for c in needs_missing:
        pct = missing_df.loc[missing_df['column']==c, 'missing_pct'].values[0]
        print(f"      {c:<40} {pct}% missing")
    print(f"  Outlier treatment        : {'⚠️  Required' if has_outliers else '✅ Clean'}")
    for c in has_outliers:
        n = outlier_df.loc[outlier_df['column']==c, 'total_outliers'].values[0]
        print(f"      {c:<40} {n} outliers")
    print(f"  High cardinality columns : {high_card or 'None'}")
    print(f"  Columns to drop          : {drop_cols}")

    print(f"\n  FEATURE ENGINEERING")
    print(f"  ─────────────────────────────────────")
    print(f"  Encoding required        : ✅  {needs_encode}")
    print(f"  Scaling required         : ⚠️  Conditional — {needs_scale}")

    print(f"\n  ADVANCED PIPELINES")
    print(f"  ─────────────────────────────────────")
    print(f"  NLP / Sentiment pipeline : ✅  {needs_nlp}")
    print(f"  RAG pipeline             : ✅  {needs_rag}")
    print(f"  Recommendation engine    : ✅  {needs_rec}")

    print(f"\n  SUMMARY")
    print(f"  ─────────────────────────────────────")
    print(f"  Regression ready         : ❌  (after cleaning → ✅)")
    print(f"  Classification ready     : ❌  (bin rate → ✅)")
    print(f"  Recommendation ready     : ✅  (cuisines, location, rest_type)")
    print(f"  NLP / Sentiment ready    : ✅  (reviews_list, dish_liked)")
    print(f"  RAG ready                : ✅  (menu_item)")
    print("=" * 55)

ml_readiness(df, missing_df, feature_df, outlier_df)

  ML READINESS REPORT

  TARGET VARIABLE
  ─────────────────────────────────────
  Column                : rate
  Type                  : Continuous (regression) / Binned (classification)
  Ready for modelling   : ❌  Needs cleaning first (rate stored as '4.1/5')

  DATA QUALITY
  ─────────────────────────────────────
  Missing value treatment  : ✅ Required
      rate                                     19.65% missing
      phone                                    2.35% missing
      location                                 0.04% missing
      rest_type                                0.44% missing
      dish_liked                               54.88% missing
      cuisines                                 0.09% missing
      approx_cost(for two people)              0.67% missing
  Outlier treatment        : ⚠️  Required
      votes                                    6880 outliers
      approx_cost(for two people)              900 outliers
  High cardinality columns : ['url']
  Columns to

In [58]:
# Cell 12 — Save Full Report to .txt
SEP  = "=" * 70
SEP2 = "-" * 70
lines = []
add = lambda *a: lines.append(" ".join(str(x) for x in a))

add(SEP)
add("  ZOMATO DATASET — PHASE 1: PROFILING REPORT (v2)")
add(f"  Generated  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
add(f"  File       : {FILE_PATH}  |  {df.shape[0]:,} rows × {df.shape[1]} columns")
add(f"  CSV size   : {CSV_MB:.2f} MB   |   In-memory: {MEM_MB:.2f} MB")
add(SEP)
add("")

# Section 1: Overview
add("SECTION 1 — DATASET OVERVIEW")
add(SEP2)
add(f"  Rows               : {len(df):,}")
add(f"  Columns            : {len(df.columns)}")
add(f"  CSV size           : {CSV_MB:.2f} MB")
add(f"  In-memory size     : {MEM_MB:.2f} MB")
add(f"  Duplicate rows     : {dup_rows:,}  ({round(dup_rows/len(df)*100,2)}%)")
add(f"  Fully null rows    : {fully_null_rows:,}")
add(f"  Fully null columns : {fully_null_cols or 'None'}")
add(f"  dtypes             : {df.dtypes.astype(str).value_counts().to_dict()}")
add("")

# Section 2: Missing Classification
add("SECTION 2 — MISSING VALUE CLASSIFICATION")
add(SEP2)
hdr = f"{'Column':<35} {'Null':>6} {'Blank':>6} {'Invalid':>8} {'Missing%':>9} {'Usable':>8}  Recommendation"
add(hdr)
add("-" * len(hdr))
for _, r in missing_df.sort_values("missing_pct", ascending=False).iterrows():
    add(f"{r['column']:<35} {r['null']:>6} {r['blank']:>6} {r['invalid_sentinel']:>8} "
        f"{r['missing_pct']:>8}% {r['usable_values']:>8}  {r['recommendation']}")
add("")

# Section 3: Feature Categories
add("SECTION 3 — FEATURE CATEGORIES & CARDINALITY")
add(SEP2)
hdr2 = f"{'Column':<35} {'Category':<22} {'Unique':>7} {'Card%':>6} {'HighCard':>9}  Action"
add(hdr2)
add("-" * len(hdr2))
for _, r in feature_df.iterrows():
    add(f"{r['column']:<35} {r['category']:<22} {r['unique_count']:>7} "
        f"{r['cardinality_pct']:>5}% {r['high_cardinality']:>9}  {r['suggested_action']}")
add("")

# Section 4: Numeric Stats (whitelist only)
add("SECTION 4 — NUMERIC STATISTICS (WHITELIST COLUMNS ONLY)")
add(SEP2)
for col, row in numeric_stats_df.iterrows():
    add(f"  COLUMN: {col}")
    for k, v in row.items():
        add(f"    {k:<15}: {v}")
    add("")

# Section 5: Outlier Detection
add("SECTION 5 — OUTLIER DETECTION (IQR METHOD)")
add(SEP2)
hdr3 = f"{'Column':<35} {'Q1':>8} {'Q3':>8} {'IQR':>8} {'LowerF':>9} {'UpperF':>9} {'OutLow':>7} {'OutHigh':>8} {'Total':>6} {'Out%':>6}"
add(hdr3)
add("-" * len(hdr3))
for _, r in outlier_df.iterrows():
    add(f"{r['column']:<35} {r['Q1']:>8} {r['Q3']:>8} {r['IQR']:>8} "
        f"{r['lower_fence']:>9} {r['upper_fence']:>9} {r['outliers_low']:>7} "
        f"{r['outliers_high']:>8} {r['total_outliers']:>6} {r['outlier_pct']:>5}%")
add("")

# Section 6: ML Readiness
add("SECTION 6 — ML READINESS REPORT")
add(SEP2)
add("  Target Variable       : rate (regression / classification)")
add("  Missing Treatment     : Required")
add("  Outlier Treatment     : Required")
add("  Encoding Required     : Yes (Categorical + Binary columns)")
add("  Scaling Required      : Conditional (Numerical columns)")
add("  NLP Pipeline          : Yes (reviews_list, dish_liked)")
add("  RAG Pipeline          : Yes (menu_item)")
add("  Recommendation Engine : Yes (cuisines, location, rest_type)")
add("  Regression Ready      : No  → After cleaning: Yes")
add("  Classification Ready  : No  → After binning rate: Yes")
add("  Recommendation Ready  : Yes")
add("")

add(SEP)
add("  END OF REPORT — v2")
add(SEP)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f"Report saved → {OUTPUT_PATH}")

Report saved → profiling_report_ipy.txt
